# 02c — Probability, statistics, and uncertainty — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/02.md) · [Course map](../PLAN.md) · [Project](../../projects/stage02/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Compute sample statistics
- Apply conditional probability and Bayes rule
- Estimate uncertainty with reproducible bootstrap samples


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

A random variable maps outcomes to numbers. Bernoulli models one binary trial; binomial counts successes across independent trials; a normal distribution is continuous and symmetric. Independence means observing one variable does not change the distribution of another. Conditional probability P(A|B) restricts attention to B. Bayes rule reverses conditioning using a base rate.

The mean is the arithmetic center; median resists extreme outliers; mode is the most frequent value. Variance measures squared deviations; standard deviation restores the original units. Sample variance uses n-1 when estimating population variance from an IID sample. Covariance tracks joint variation; correlation normalizes it and does not establish causation.

A confidence interval is a property of a repeated-sampling procedure, not the probability that a fixed parameter lies in this one observed interval. Bootstrap intervals resample observed data and inherit its biases. A p-value is the probability, under a specified null model, of a statistic at least as extreme as observed; it is not the probability the null is true.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import statistics
sample = [2, 4, 4, 6]
print(statistics.mean(sample), statistics.median(sample), statistics.variance(sample))


## Exercise 1: Sample variance

Compute unbiased sample variance for n >= 2; reject smaller samples.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def variance(xs):
    if len(xs) < 2: raise ValueError("Need two observations")
    mean = sum(xs)/len(xs)
    return sum((x-mean)**2 for x in xs)/(len(xs)-1)


In [ ]:
assert variance([1, 2, 3]) == 1
expect_error(ValueError, lambda: variance([1]))
print("Exercise 1: checks passed")


**Why this works:** The n-1 adjustment compensates for estimating the mean from the same sample.


## Exercise 2: Bayesian alert

Return P(event|positive) from prevalence, sensitivity, and false_positive_rate in [0,1]. Reject an impossible positive event.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def posterior(prevalence, sensitivity, false_positive_rate):
    if any(not 0 <= p <= 1 for p in (prevalence, sensitivity, false_positive_rate)):
        raise ValueError("Invalid probability")
    true = prevalence*sensitivity
    positive = true + (1-prevalence)*false_positive_rate
    if positive == 0: raise ValueError("Impossible conditioning event")
    return true/positive


In [ ]:
assert abs(posterior(.01, .9, .1)-(.009/.108)) < 1e-12
expect_error(ValueError, lambda: posterior(0, 1, 0))
print("Exercise 2: checks passed")


**Why this works:** Rare events can yield many false alarms even with high sensitivity.


## Exercise 3: Bootstrap mean interval

Return the 2.5th and 97.5th percentile order statistics of 1000 bootstrap means using seed=42. Require nonempty data.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def bootstrap_ci(xs, repeats=1000, seed=42):
    import random
    if not xs or repeats < 40: raise ValueError("Insufficient data or repeats")
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(xs, k=len(xs)))/len(xs) for _ in range(repeats))
    return means[int(.025*repeats)], means[int(.975*repeats)]


In [ ]:
lo, hi = bootstrap_ci([1, 2, 3, 4, 5])
assert lo <= 3 <= hi
assert bootstrap_ci([2]*10) == (2, 2)
print("Exercise 3: checks passed")


**Why this works:** This simple percentile interval assumes representative independent observations; time series need block-aware methods.


## Independent transfer

Simulate 10,000 Bernoulli trials, plot a binomial count histogram, calculate percentiles/covariance/correlation, and run a permutation test of a mean difference. Explain sampling bias and multiple testing.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. What is a base-rate error?

   Ignoring prevalence when converting a test result to a posterior probability.

2. Why is correlation insufficient for causation?

   Confounding, selection, or reverse causality can produce association.


## Reading and review

- [Primary reference 1](https://d2l.ai/chapter_preliminaries/probability.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
